Unbiased Model Creation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from scipy.stats import uniform

# Load the dataset
df = pd.read_csv('/Users/bapbap23/Desktop/Patient-No-Show-prediction-/patients.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/bapbap23/Desktop/Patient-No-Show-prediction-/patients_new.csv'

In [2]:
df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,...,Handcap,SMS_received,No-show,ScheduledHour,AppointmentHour,DaysBetween,ScheduledWeekday,AppointmentWeekday,AppointmentMonth,No-show_numeric
0,2.987250e+13,5642903,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,...,0,0,No,18,0,0,4,4,4,0
1,5.589978e+14,5642503,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,...,0,0,No,16,0,0,4,4,4,0
2,4.262962e+12,5642549,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,...,0,0,No,16,0,0,4,4,4,0
3,8.679512e+11,5642828,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,...,0,0,No,17,0,0,4,4,4,0
4,8.841186e+12,5642494,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,...,0,0,No,16,0,0,4,4,4,0


In [ ]:
df['No-show_numeric'] = df['No-show'].apply(lambda x: 1 if x == 'Yes' else 0)

In [ ]:
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

In [ ]:
df['ScheduledHour'] = df['ScheduledDay'].dt.hour
df['AppointmentHour'] = df['AppointmentDay'].dt.hour
df['ScheduledWeekday'] = df['ScheduledDay'].dt.dayofweek  # 0=Monday
df['AppointmentWeekday'] = df['AppointmentDay'].dt.dayofweek
df['AppointmentMonth'] = df['AppointmentDay'].dt.month

In [ ]:
df['DaysBetween'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days
df['DaysBetween'] = df['DaysBetween'].clip(lower=0)

In [ ]:
bins = [-1, 0, 7, 30, df['DaysBetween'].max()]
labels = ['0_days', '1-7_days', '8-30_days', '>30_days']
df['DaysBetween_binned'] = pd.cut(df['DaysBetween'], bins=bins, labels=labels, right=True)

In [ ]:
df['SMS_DaysBetween'] = df['SMS_received'] * df['DaysBetween']
df['Age_Hipertension'] = df['Age'] * df['Hipertension']

In [ ]:
neighbourhood_target_encoding = df.groupby('Neighbourhood')['No-show_numeric'].transform('mean')
df['Neighbourhood_Encoded'] = neighbourhood_target_encoding

In [ ]:
df['Weekday_Diff'] = (df['AppointmentWeekday'] - df['ScheduledWeekday'] + 7) % 7

In [ ]:
unbiased_features_final = ['Age', 'DaysBetween', 'SMS_received', 'ScheduledWeekday',
                           'AppointmentWeekday', 'AppointmentHour', 'ScheduledHour', 'AppointmentMonth',
                           'DaysBetween_binned', 'SMS_DaysBetween', 'Age_Hipertension', 'Neighbourhood_Encoded', 'Weekday_Diff']

In [ ]:
X_unbiased_final = df[unbiased_features_final]
y_unbiased_final = df['No-show_numeric']

In [ ]:
categorical_features_to_encode_unbiased = ['ScheduledWeekday', 'AppointmentWeekday', 'AppointmentMonth', 'DaysBetween_binned']
X_unbiased_final = pd.get_dummies(X_unbiased_final, columns=categorical_features_to_encode_unbiased, drop_first=True)

In [ ]:
X_train_unbiased_final, X_test_unbiased_final, y_train_unbiased_final, y_test_unbiased_final = train_test_split(X_unbiased_final, y_unbiased_final, test_size=0.2, random_state=42)
numerical_cols_unbiased_final = X_train_unbiased_final.select_dtypes(include=np.number).columns.tolist()

# 23. Scale unbiased numerical features
scaler_unbiased_final = StandardScaler()
X_train_unbiased_final_scaled = scaler_unbiased_final.fit_transform(X_train_unbiased_final[numerical_cols_unbiased_final])
X_test_unbiased_final_scaled = scaler_unbiased_final.transform(X_test_unbiased_final[numerical_cols_unbiased_final])

# 24. Apply SMOTE to unbiased training data
smote_unbiased = SMOTE(random_state=42)
X_train_unbiased_resampled_final, y_train_unbiased_resampled_final = smote_unbiased.fit_resample(X_train_unbiased_final_scaled, y_train_unbiased_final)

In [ ]:
random_search_unbiased_final = RandomizedSearchCV(lgb.LGBMClassifier(random_state=42),
                                            param_distributions=param_dist,
                                            n_iter=50,
                                            scoring='accuracy',
                                            cv=3,
                                            verbose=1,
                                            random_state=42,
                                            n_jobs=-1)

In [ ]:
print("\nStarting RandomizedSearchCV for unbiased LightGBM model...")
random_search_unbiased_final.fit(X_train_unbiased_resampled_final, y_train_unbiased_resampled_final)

# 31. Get the best unbiased model
best_lgbm_unbiased_final = random_search_unbiased_final.best_estimator_
print("\nBest parameters for unbiased LightGBM model:")
print(random_search_unbiased_final.best_params_)

In [ ]:
print("Best Unbiased LightGBM Model Evaluation (with SMOTE, all New Features, and Tuned Hyperparameters):")
y_pred_best_lgbm_unbiased_final = best_lgbm_unbiased_final.predict(X_test_unbiased_final_scaled)
print("Accuracy:", accuracy_score(y_test_unbiased_final, y_pred_best_lgbm_unbiased_final))
print("Classification Report:\n", classification_report(y_test_unbiased_final, y_pred_best_lgbm_unbiased_final))
print("Confusion Matrix:\n", confusion_matrix(y_test_unbiased_final, y_pred_best_lgbm_unbiased_final))